# Learned possession: first Kaggle experiment

Attach the private match dataset and learned-possession-source.zip as notebook inputs. This first stage is CPU-based; a GPU is not required. The later temporal-network experiment will use GPU time.

Default mode is a public-window smoke test, not evidence of generalisation or a tuned competition submission. Change to train only after supplying separate training and validation annotation files. No prediction or annotation files are uploaded by this notebook.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile
import json
import hashlib

MODE = "public-smoke"  # "prepare", "public-smoke", or "train"
CODE_DIR = None        # Optional Path to an unpacked source folder
DATA_DIR = None        # Optional Path to the folder containing tactical/
TRAIN_LABELS = None   # Full-length CSV, X outside training windows
VALIDATION_LABELS = None  # Separate full-length CSV; no overlap
WORK = Path("/kaggle/working/possession_experiment")
FEATURES = WORK / "features.npz"
RUN = WORK / ("public_smoke" if MODE == "public-smoke" else "run01")
WORK.mkdir(parents=True, exist_ok=True)
assert sys.version_info >= (3, 11), "The pinned experiment needs Python 3.11+."
assert MODE in {"prepare", "public-smoke", "train"}


In [ ]:
INPUT = Path("/kaggle/input")
if CODE_DIR is None:
    roots = [p.parent for p in INPUT.rglob("requirements-learning.txt")
             if (p.parent / "learned_possession" / "__main__.py").is_file()]
    if len(roots) == 1:
        CODE_DIR = roots[0]
    else:
        archives = list(INPUT.rglob("learned-possession-source.zip"))
        if len(archives) != 1:
            raise ValueError("Attach one learned-possession-source.zip, or set CODE_DIR explicitly.")
        destination = WORK / "source"
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError("Source archive has a path outside its extraction folder.")
            archive.extractall(destination)
        CODE_DIR = destination / "Barca_hack"
CODE_DIR = Path(CODE_DIR)
assert (CODE_DIR / "learned_possession" / "__main__.py").is_file()
if DATA_DIR is None:
    matches = [p.parent.parent for p in INPUT.rglob("BAR-ATH_tactical_atd.txt")
               if p.parent.name == "tactical"]
    if len(matches) != 1:
        raise ValueError("Set DATA_DIR to the folder containing the intended tactical/ directory.")
    DATA_DIR = matches[0]
DATA_DIR = Path(DATA_DIR)
required = ["BAR-ATH_tactical.mp4", "BAR-ATH_tactical_atd.xml",
            "BAR-ATH_tactical_atd.txt", "BAR-ATH_tactical_homography.csv",
            "BAR-ATH_tactical_smart_tagging.xml"]
missing = [name for name in required if not (DATA_DIR / "tactical" / name).is_file()]
assert not missing, f"Missing tactical assets: {missing}"
print("Source:", CODE_DIR)
print("Data root:", DATA_DIR)
print("Output:", WORK)


Enable Internet for the initial dependency installation. The experiment uses its own pinned requirements, separately from v6's byte-reproduction stack. All subsequent commands fail visibly if a check fails.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(CODE_DIR / "requirements-learning.txt")], check=True)
COMMAND_ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=CODE_DIR,
                   env=COMMAND_ENV, check=True)
run("-m", "unittest", "discover", "-s", "tests", "-v")


Prepare observation-only features. Parser and audio caches go beside the feature file under the writable working directory, not into the read-only match dataset. Existing features are reused only when their recorded source sizes and modification times match.


In [ ]:
rebuild = not FEATURES.is_file()
if not rebuild:
    sys.path.insert(0, str(CODE_DIR))
    from learned_possession.features import FeatureCache
    saved_sources = FeatureCache.load(FEATURES).provenance.get("sources", {})
    rebuild = len(saved_sources) != len(required)
    for item in saved_sources.values():
        source = DATA_DIR / "tactical" / item["name"]
        rebuild |= (not source.is_file()
                    or source.stat().st_size != item["size"]
                    or source.stat().st_mtime_ns != item["mtime_ns"])
if rebuild:
    run("-m", "learned_possession", "prepare", "--data-dir", DATA_DIR, "--out", FEATURES)
else:
    print("Reusing matching feature cache:", FEATURES)


The public smoke uses a fixed first-three-minutes / last-two-minutes split with temporal purging. Those public labels were already used in v6 development: this is a pipeline check, not an independent model comparison. A meaningful comparison needs separately annotated windows and an untouched final holdout.


In [ ]:
if MODE != "prepare":
    command = ["-m", "learned_possession", MODE, "--features", FEATURES,
               "--out", RUN, "--threads", 4, "--bootstrap", 1000,
               "--compare", CODE_DIR / "evaluation/results/possession_v6.csv",
               CODE_DIR / "evaluation/results/baseline_smooth50.csv"]
    if MODE == "train":
        if TRAIN_LABELS is None or VALIDATION_LABELS is None:
            raise ValueError("Set TRAIN_LABELS and VALIDATION_LABELS before training.")
        command += ["--train-labels", Path(TRAIN_LABELS),
                    "--validation-labels", Path(VALIDATION_LABELS)]
    run(*command)
    result = json.loads((RUN / "report.json").read_text())
    print("Evaluation kind:", result["training"]["evaluation_kind"])
    for name, score in result["scores"].items():
        print(name, "spellF1", round(score["spell_f1_mean"], 4),
              "macroF1", round(score["macro_f1"], 4),
              "predicted/true possessions", score["n_pred_spells"], "/", score["n_gt_spells"])


Check that the saved model regenerates exactly the same CSV without reading labels. Save/download the feature cache, model, reports and versions using Kaggle's output controls before ending the session. The raw data remains an input; this notebook does not publish anything.


In [ ]:
if MODE != "prepare":
    regenerated = RUN / "regenerated.csv"
    run("-m", "learned_possession", "predict", "--features", FEATURES,
        "--model", RUN / "model.joblib", "--out", regenerated)
    original_bytes = (RUN / "learned.csv").read_bytes()
    assert regenerated.read_bytes() == original_bytes
    print("Byte-identical prediction SHA-256:", hashlib.sha256(original_bytes).hexdigest())
    print("Artifacts:", RUN)
